In [97]:
import pandas as pd 
import numpy as np 
import requests

base_url = 'https://fapi.binance.com'
kline_path = '/fapi/v1/continuousKlines'
exchange_info_path = '/fapi/v1/exchangeInfo'

def get_symbols():
    url = f"{base_url}{exchange_info_path}"
    response = requests.get(url)
    data = response.json()
    symbols = [item['symbol'] for item in data['symbols'] if item['contractType'] == 'PERPETUAL' 
               and item['status'] == 'TRADING' 
               and item['quoteAsset'] == 'USDT'
               ]

    return symbols

def infer_trend_side(pct_change, amplitude, min_body_to_range=0.35):
    """Use close-open move versus high-low range to judge who controlled the candle."""
    if amplitude == 0:
        return 'neutral'
    body_to_range = abs(pct_change) / amplitude
    if body_to_range < min_body_to_range:
        return 'neutral'
    return 'long' if pct_change > 0 else 'short'

def get_klines(symbol, interval, start_time=None, end_time=None, limit=100):
    url = f"{base_url}{kline_path}"
    params = {
        "pair": symbol,
        'contractType': 'PERPETUAL',
        "interval": interval,
        "startTime": start_time,
        "endTime": end_time,
        "limit": limit
    }
    kline = requests.get(url, params=params).json()
    
    try:
        kline_data = [{
                    'symbol': symbol,
                    'interval': interval,
                    'open': float(x[1]),
                    'high': float(x[2]),
                    'low': float(x[3]),
                    'close': float(x[4]),
                    'pct_change': round((float(x[4]) - float(x[1])) / float(x[1]) * 100, 2),
                    'amplitude': round((float(x[2]) - float(x[3])) / float(x[3]) * 100, 2),
                    'vol_usdt': float(x[7]),
                    'active_buy_ratio': 0 if float(x[5]) == 0 else round(float(x[9]) / float(x[5]) * 100, 2),
                    'active_sell_ratio': 0 if float(x[5]) == 0 else round((float(x[5]) - float(x[9])) / float(x[5]) * 100, 2),
                    'open_time': pd.to_datetime(x[0], unit='ms'),
                    'open_ts': x[0],
                    'close_ts':x[6]
                    } for x in kline]
        prev_vol_usdt = None
        for item in kline_data:
            item['body_to_range'] = 0 if item['amplitude'] == 0 else round(abs(item['pct_change']) / item['amplitude'], 2)
            item['trend_side'] = infer_trend_side(item['pct_change'], item['amplitude'])
            item['vol_ratio'] = None if not prev_vol_usdt else round(item['vol_usdt'] / prev_vol_usdt, 2)
            prev_vol_usdt = item['vol_usdt']
        return kline_data
    except Exception as e:
        print(f"Error processing kline data for symbol {symbol}: {e}")
        return kline 

In [106]:
AMPLITUDE_MIN = 30
BODY_TO_RANGE_MIN = 0.35
VOL_RATIO_MIN = 1.5

large_move_symbols = pd.DataFrame()
for symbol in get_symbols()[-50:]:
    kline = get_klines(symbol, '1d', limit=7)
    kline = [
        x for x in kline
        if abs(x['amplitude']) > AMPLITUDE_MIN
        and x['body_to_range'] >= BODY_TO_RANGE_MIN
        and x['trend_side'] != 'neutral'
        and (x['vol_ratio'] is not None and x['vol_ratio'] >= VOL_RATIO_MIN)
    ]
    if kline:
        for entry in kline:
            print(
                f'get trend candidate symbol:{symbol}, side:{entry["trend_side"]}, '
                f'amplitude:{entry["amplitude"]}%, pct_change:{entry["pct_change"]}%, '
                f'body_to_range:{entry["body_to_range"]}, vol_ratio:{entry["vol_ratio"]}, '
                f'open_time:{entry["open_time"]}'
            )
        temp = pd.DataFrame(kline)
        large_move_symbols = pd.concat([large_move_symbols, temp], ignore_index=True)

large_move_symbols.sort_values(
    ['vol_ratio', 'body_to_range', 'amplitude'],
    ascending=False,
    na_position='last'
).reset_index(drop=True)


get trend candidate symbol:USUSDT, side:short, amplitude:32.03%, pct_change:-17.33%, body_to_range:0.54, vol_ratio:1.92, open_time:2026-05-11 00:00:00
get trend candidate symbol:GUAUSDT, side:long, amplitude:33.24%, pct_change:30.15%, body_to_range:0.91, vol_ratio:5.75, open_time:2026-05-12 00:00:00
get trend candidate symbol:GWEIUSDT, side:long, amplitude:32.46%, pct_change:11.85%, body_to_range:0.37, vol_ratio:5.5, open_time:2026-05-14 00:00:00
get trend candidate symbol:AIGENSYNUSDT, side:long, amplitude:82.12%, pct_change:36.26%, body_to_range:0.44, vol_ratio:4.79, open_time:2026-05-14 00:00:00


,symbol,interval,open,high,low,close,pct_change,amplitude,vol_usdt,active_buy_ratio,active_sell_ratio,open_time,open_ts,close_ts,body_to_range,trend_side,vol_ratio
0,GUAUSDT,1d,1.009500,1.333300,1.000700,1.313900,30.15,33.24,4.113722e+07,50.49,49.51,2026-05-12,1778544000000,1778630399999,0.91,long,5.75
1,GWEIUSDT,1d,0.131570,0.161390,0.121840,0.147160,11.85,32.46,2.336500e+07,52.00,48.00,2026-05-14,1778716800000,1778803199999,0.37,long,5.50
2,AIGENSYNUSDT,1d,0.029230,0.052960,0.029080,0.039830,36.26,82.12,4.831404e+08,50.15,49.85,2026-05-14,1778716800000,1778803199999,0.44,long,4.79
3,USUSDT,1d,0.006935,0.007292,0.005523,0.005733,-17.33,32.03,8.962990e+07,49.51,50.49,2026-05-11,1778457600000,1778543999999,0.54,short,1.92


In [107]:
get_symbol = large_move_symbols.symbol.iloc[1]
import numpy as np 
np.where(large_move_symbols['symbol'] == get_symbol)
large_move_symbols[large_move_symbols['symbol'] == get_symbol]

,symbol,interval,open,high,low,close,pct_change,amplitude,vol_usdt,active_buy_ratio,active_sell_ratio,open_time,open_ts,close_ts,body_to_range,trend_side,vol_ratio
1,GUAUSDT,1d,1.0095,1.3333,1.0007,1.3139,30.15,33.24,4.113722e+07,50.49,49.51,2026-05-12,1778544000000,1778630399999,0.91,long,5.75


In [108]:
large_move_symbols

,symbol,interval,open,high,low,close,pct_change,amplitude,vol_usdt,active_buy_ratio,active_sell_ratio,open_time,open_ts,close_ts,body_to_range,trend_side,vol_ratio
0,USUSDT,1d,0.006935,0.007292,0.005523,0.005733,-17.33,32.03,8.962990e+07,49.51,50.49,2026-05-11,1778457600000,1778543999999,0.54,short,1.92
1,GUAUSDT,1d,1.009500,1.333300,1.000700,1.313900,30.15,33.24,4.113722e+07,50.49,49.51,2026-05-12,1778544000000,1778630399999,0.91,long,5.75
2,GWEIUSDT,1d,0.131570,0.161390,0.121840,0.147160,11.85,32.46,2.336500e+07,52.00,48.00,2026-05-14,1778716800000,1778803199999,0.37,long,5.50
3,AIGENSYNUSDT,1d,0.029230,0.052960,0.029080,0.039830,36.26,82.12,4.831404e+08,50.15,49.85,2026-05-14,1778716800000,1778803199999,0.44,long,4.79


In [109]:
test = pd.DataFrame(get_klines('GUAUSDT', '1m',limit=1000))

In [112]:
AMPLITUDE_MIN = 3
BODY_TO_RANGE_MIN = 0.6
VOL_RATIO_MIN = 2 

test[(test['amplitude']>= AMPLITUDE_MIN) & (test['body_to_range'] >= BODY_TO_RANGE_MIN) & (test['vol_ratio'] >= VOL_RATIO_MIN)]


,symbol,interval,open,high,low,close,pct_change,amplitude,vol_usdt,active_buy_ratio,active_sell_ratio,open_time,open_ts,close_ts,body_to_range,trend_side,vol_ratio
61,GUAUSDT,1m,1.3068,1.3074,1.2424,1.2600,-3.58,5.23,684912.63500,43.17,56.83,2026-05-15 12:40:00,1778848800000,1778848859999,0.68,short,3.20
68,GUAUSDT,1m,1.2861,1.3352,1.2861,1.3192,2.57,3.82,100406.75170,58.66,41.34,2026-05-15 12:47:00,1778849220000,1778849279999,0.67,long,2.14
130,GUAUSDT,1m,1.2076,1.2100,1.1365,1.1540,-4.44,6.47,363928.38525,44.92,55.08,2026-05-15 13:49:00,1778852940000,1778852999999,0.69,short,2.69
198,GUAUSDT,1m,1.2720,1.3360,1.2710,1.3281,4.41,5.11,401857.82450,54.44,45.56,2026-05-15 14:57:00,1778857020000,1778857079999,0.86,long,6.19


,symbol,interval,open,high,low,close,pct_change,amplitude,vol_usdt,active_buy_ratio,active_sell_ratio,open_time,open_ts,close_ts,body_to_range,trend_side,vol_ratio
